# The Azure Stay High Distribution Costs - Generate Data

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import random
import os

from datetime import datetime, timedelta

## Configuration

In [2]:
# Set random seed
np.random.seed(42)
random.seed(42)

In [3]:
# Parameters
YEAR = 2025
START_DATE = datetime(YEAR, 1, 1)
END_DATE = datetime(YEAR, 12, 31)
NUM_BOOKINGS = 8000
NUM_GUESTS = 2000

In [4]:
# 1. Setup Directory
output_dir = './Data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'AzureStay_Data_2025.xlsx')

## Generate Data

In [5]:
def random_date(start, end):
    return start + timedelta(days=random.randint(0, (end - start).days))

# 1. dim_channels
channels_data = [
    ('CH_01', 'Booking.com', 'OTA', 'Percentage', 0.15, 'John Doe', 0.0),
    ('CH_02', 'Expedia', 'OTA', 'Percentage', 0.18, 'John Doe', 0.0),
    ('CH_03', 'Direct Web', 'Direct', 'Net Rate', 0.00, 'Jane Smith', 0.0), 
    ('CH_04', 'Agoda', 'OTA', 'Percentage', 0.15, 'John Doe', 0.0),
    ('CH_05', 'Walk-in', 'Direct', 'Net Rate', 0.00, 'Jane Smith', 0.0),
    ('CH_06', 'Corporate GDS', 'Wholesale', 'Flat Fee', 0.00, 'Alice Brown', 10.0)
]
dim_channels = pd.DataFrame(channels_data, columns=[
    'channel_id', 'channel_name', 'channel_type', 'commission_model', 
    'default_commission_rate', 'contract_owner', 'default_flat_fee_amount'
])

# 2. dim_rate_codes
rates_data = [
    ('RT_RACK', 'Standard Rack Rate', 'Base price no discount', True),
    ('RT_PROMO', 'Summer Promo', '10% off base price', True),
    ('RT_CORP', 'Corporate Flat Rate', 'Fixed rate for corporate', False),
    ('RT_MEMBER', 'Direct Member Rate', 'Includes breakfast & Wifi', False)
]
dim_rate_codes = pd.DataFrame(rates_data, columns=['rate_code_id', 'rate_name', 'description', 'is_commissionable'])

# 3. dim_segments
segments_data = [
    ('FIT', 'Leisure', 'B2C', 'Standard individual travelers'),
    ('CORP', 'Corporate', 'B2B', 'Special corporate rate for partnered companies'),
    ('GRP', 'Group', 'B2B', 'Groups of 10+ rooms'),
    ('WSL', 'Wholesaler', 'B2B', 'Travel agents net rates')
]
dim_segments = pd.DataFrame(segments_data, columns=['segment_id', 'segment_name', 'segment_category', 'description'])

# 4. dim_room_types
rooms_data = [
    ('RT_STD', 'Standard Queen', 2, 50, 100), 
    ('RT_DLX', 'Deluxe King', 2, 30, 150),
    ('RT_STE', 'Suite', 4, 10, 300)
]
dim_room_types = pd.DataFrame(rooms_data, columns=['room_type_id', 'room_type_name', 'base_capacity', 'total_inventory_count', 'base_price'])
base_prices = dict(zip(dim_room_types['room_type_id'], dim_room_types['base_price']))
dim_room_types_clean = dim_room_types.drop(columns=['base_price'])

# 5. dim_guests
guests = []
for i in range(1, NUM_GUESTS + 1):
    guests.append({
        'guest_id': f'G-{i:04d}',
        'full_name': f"Guest_Name_{i}",
        'nationality': random.choices(['TH', 'US', 'UK', 'JP', 'SG', 'CN', 'AU'], weights=[0.4, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1])[0],
        'guest_type': random.choices(['New Guest', 'Returning Guest'], weights=[0.7, 0.3])[0]
    })
dim_guests = pd.DataFrame(guests)

# 6. dim_date
date_range = pd.date_range(start=START_DATE, end=END_DATE)
dim_date = pd.DataFrame({'date': date_range.date})
holidays = set(random.sample(list(dim_date['date']), 15))
dim_date['is_holiday'] = dim_date['date'].apply(lambda x: x in holidays)

# 7. fact_daily_inventory
inventory_list = []
for d in dim_date['date']:
    for _, row in dim_room_types_clean.iterrows():
        ooo = random.choices([0, 1, 2], weights=[0.8, 0.15, 0.05])[0] # Out of order
        inventory_list.append({
            'date': d,
            'room_type_id': row['room_type_id'],
            'available_rooms': row['total_inventory_count'] - ooo,
            'out_of_order_rooms': ooo
        })
fact_daily_inventory = pd.DataFrame(inventory_list)

# 8. fact_marketing_spend
spend_list = []
spend_id = 1
for month in range(1, 13):
    # Google Ads & Facebook for Direct Web
    for plat in ['Google Ads', 'Facebook']:
        spend_list.append({
            'spend_id': f'SP_{spend_id:03d}',
            'channel_id': 'CH_03',
            'spend_date': datetime(YEAR, month, random.randint(1, 28)).date(),
            'platform': plat,
            'cost_amount': round(random.uniform(300, 1500), 2),
            'clicks': random.randint(800, 5000)
        })
        spend_id += 1
fact_marketing_spend = pd.DataFrame(spend_list)

# 9. fact_bookings (8000 Rows, Consistency logic applied)
bookings = []
statuses = ['Confirmed', 'Checked-Out', 'Cancelled', 'No-Show']
for i in range(1, NUM_BOOKINGS + 1):
    check_in = random_date(START_DATE, END_DATE - timedelta(days=1))
    stay_length = random.randint(1, 7)
    check_out = check_in + timedelta(days=stay_length)
    booking_date = check_in - timedelta(days=random.randint(1, 60))
    
    channel = dim_channels.sample(1, weights=[0.3, 0.25, 0.2, 0.15, 0.05, 0.05]).iloc[0]
    room_type = dim_room_types.sample(1, weights=[0.6, 0.3, 0.1]).iloc[0]
    
    rate_id = 'RT_PROMO' if channel['channel_id'] == 'CH_03' else random.choices(['RT_RACK', 'RT_PROMO'], weights=[0.6, 0.4])[0]
    segment_id = 'CORP' if channel['channel_type'] == 'Wholesale' else 'FIT'
    
    num_rooms = random.choices([1, 2, 3], weights=[0.85, 0.1, 0.05])[0]
    
    # Calculate Revenue
    gross_revenue = round(base_prices[room_type['room_type_id']] * stay_length * num_rooms * (0.9 if rate_id == 'RT_PROMO' else 1.0), 2)
    
    # Calculate Commission Consistency
    if channel['commission_model'] == 'Percentage':
        comm_amount = round(gross_revenue * channel['default_commission_rate'], 2)
    elif channel['commission_model'] == 'Flat Fee':
        comm_amount = round(channel['default_flat_fee_amount'] * num_rooms, 2)
    else:
        comm_amount = 0.0
        
    net_revenue = gross_revenue - comm_amount
    ancillary = round(random.uniform(0, 200), 2) if random.random() > 0.5 else 0.0
    
    bookings.append({
        'booking_id': f'RES-{i:05d}',
        'guest_id': random.choice(guests)['guest_id'],
        'channel_id': channel['channel_id'],
        'room_type_id': room_type['room_type_id'],
        'rate_code_id': rate_id,
        'segment_id': segment_id,
        'booking_date': booking_date.date(),
        'check_in_date': check_in.date(),
        'check_out_date': check_out.date(),
        'gross_room_revenue': gross_revenue,
        'commission_amount': comm_amount,
        'net_room_revenue': net_revenue,
        'status': random.choices(statuses, weights=[0.1, 0.7, 0.15, 0.05])[0],
        'adults_count': random.randint(1, room_type['base_capacity']),
        'children_count': random.randint(0, 2),
        'number_of_rooms': num_rooms,
        'ancillary_revenue': ancillary
    })

fact_bookings = pd.DataFrame(bookings)

# Final Output saving
with pd.ExcelWriter(file_path) as writer:
    fact_bookings.to_excel(writer, sheet_name='fact_bookings', index=False)
    dim_channels.to_excel(writer, sheet_name='dim_channels', index=False)
    dim_rate_codes.to_excel(writer, sheet_name='dim_rate_codes', index=False)
    fact_marketing_spend.to_excel(writer, sheet_name='fact_marketing_spend', index=False)
    dim_guests.to_excel(writer, sheet_name='dim_guests', index=False)
    dim_segments.to_excel(writer, sheet_name='dim_segments', index=False)
    dim_room_types_clean.to_excel(writer, sheet_name='dim_room_types', index=False)
    dim_date.to_excel(writer, sheet_name='dim_date', index=False)
    fact_daily_inventory.to_excel(writer, sheet_name='fact_daily_inventory', index=False)